# Entraînement de agbledɔ01

## Environnement et dépendances

In [1]:
import os
import zipfile
from pathlib import Path
import yaml
import shutil
from tqdm import tqdm

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
    print("Environnement Google Colab détecté.")
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01')
    os.chdir(PROJECT_ROOT)

except ImportError:
    IN_COLAB = False
    print("Environnement local détecté.")
    PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
    os.chdir(PROJECT_ROOT)

print(f"Racine du projet : {PROJECT_ROOT}")

Environnement Google Colab détecté.
Mounted at /content/drive
Racine du projet : /content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01


In [ ]:
if IN_COLAB:
    %pip install -q ultralytics mlflow

import mlflow
from ultralytics import YOLO
from ultralytics import settings

#settings.update({"mlflow": False})
# Configuration de MLflow
#print("Initialisation du tracking MLflow...")
#mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_ROOT / 'mlruns' / 'mlflow.db').as_posix()}")
#mlflow.set_experiment("agbledɔ01")
#print("✅ MLflow : prêt.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87

## Data et configs

In [4]:
model_config_path = PROJECT_ROOT / "configs/model.yaml"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

with open(str(model_config_path), "r") as f:
    hyp_config = yaml.safe_load(f)

mlflow.set_tracking_uri(f"sqlite:///{(Path('mlruns') / 'agbledɔ01.db').as_posix()}")
settings.update({"mlflow": True})

In [5]:
if IN_COLAB:

    # ========= Config =========
    drive_zip = str(PROJECT_ROOT / "data" / "processed.zip")
    colab_zip = "/content/processed.zip"
    COLAB_DATA_DIR = "/content/data"

    # ========= Utils =========
    def zip_valid(zip_path):
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                return z.testzip() is None
        except:
            return False

    def total_size(path):
        return sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(path)
            for f in files
        )

    # ========= Build zip if needed =========
    if not os.path.exists(drive_zip) or not zip_valid(drive_zip):

        print("\n📦 Création du zip...")

        if os.path.exists(drive_zip):
            os.remove(drive_zip)

        with zipfile.ZipFile(
            drive_zip,
            "w",
            zipfile.ZIP_DEFLATED
        ) as z:

            for root, _, files in os.walk(str(DATA_DIR)):
                for file in tqdm(files):
                    full = os.path.join(root, file)
                    rel = os.path.relpath(full, str(DATA_DIR))
                    z.write(full, rel)

        if not zip_valid(drive_zip):
            raise Exception("❌ ZIP invalide")

        print("[OK] - ZIP OK")

    else:
        print("\n[OK] - ZIP déjà valide")

    # ========= Copy =========
    print("\n🚚 Copie vers /content...")

    if os.path.exists(colab_zip):
        os.remove(colab_zip)

    shutil.copy2(drive_zip, colab_zip)
    print("[OK] - Copie")

    # ========= Extract =========
    print("\n📂 Extraction...")

    if os.path.exists(COLAB_DATA_DIR):
        shutil.rmtree(COLAB_DATA_DIR)

    os.makedirs(COLAB_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(colab_zip, "r") as z:
        z.extractall(COLAB_DATA_DIR)

    print("[OK] - Extraction")

    # ========= Verify =========
    print("\n🔎 Vérification...")

    src_count = sum(
        len(files)
        for _, _, files in os.walk(str(DATA_DIR))
    )

    dst_count = sum(
        len(files)
        for _, _, files in os.walk(COLAB_DATA_DIR)
    )

    src_size = total_size(str(DATA_DIR))
    dst_size = total_size(COLAB_DATA_DIR)

    assert src_count == dst_count, "❌ Nombre fichiers différent"
    assert src_size == dst_size, "❌ Taille dataset différente"

    print("[OK] - Vérification")

    # ========= Cleanup =========
    os.remove(colab_zip)
    DATA_DIR = Path(COLAB_DATA_DIR)
    print("\n [OK] - Dataset prêt :", DATA_DIR)


📦 Création du zip...


100%|██████████| 1/1 [00:00<00:00,  2.57it/s]
0it [00:00, ?it/s]
100%|██████████| 843/843 [00:15<00:00, 53.80it/s] 
0it [00:00, ?it/s]
100%|██████████| 298/298 [00:04<00:00, 62.24it/s] 


[OK] - ZIP OK

🚚 Copie vers /content...
[OK] - Copie

📂 Extraction...
[OK] - Extraction

🔎 Vérification...
[OK] - Vérification

 [OK] - Dataset prêt : /content/data


## Lancement et Reprise


In [6]:
run_name = "agbledɔ01"
run_dir = PROJECT_ROOT / "runs/classify" / run_name
last_checkpoint = run_dir / "weights" / "last.pt"

if last_checkpoint.exists() and last_checkpoint.stat().st_size > 0:
    print(f"🔄 Reprise de l'entraînement détectée : {last_checkpoint}\n")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    print("▶️ Nouvel entraînement.\n")
    model = YOLO(str(MODELS_DIR / hyp_config["model"]))
    results = model.train(
        data=str(DATA_DIR),
        name=run_name,
        project=f"{run_name}-yolo_experiment-01",
        #resume=resume_flag,
        epochs=hyp_config["epochs"],
        patience=hyp_config["patience"],
        batch=hyp_config["batch"],
        imgsz=hyp_config["imgsz"],
        device=hyp_config["device"],
        optimizer=hyp_config["optimizer"],
        lr0=hyp_config["lr0"],
        lrf=hyp_config["lrf"],
        weight_decay=hyp_config["weight_decay"],
        hsv_h=hyp_config["hsv_h"],
        hsv_s=hyp_config["hsv_s"],
        hsv_v=hyp_config["hsv_v"],
        degrees=hyp_config["degrees"],
        translate=hyp_config["translate"],
        scale=hyp_config["scale"],
        fliplr=hyp_config["fliplr"],
        mosaic=hyp_config["mosaic"],
        deterministic=True,
        seed=42,
        save=True
    )

▶️ Nouvel entraînement.

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01/models/yolo11s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=agbledɔ01, nbs=64, nms=False, 

2026/05/11 12:18:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/11 12:18:53 INFO mlflow.store.db.utils: Updating database tables
2026/05/11 12:18:57 INFO mlflow.tracking.fluent: Experiment with name 'agbledɔ01-yolo_experiment-01' does not exist. Creating a new experiment.
2026/05/11 12:18:57 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/11 12:18:57 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(3c24a222a4a748d6a7e4f4b0832a1320) to sqlite:///mlruns/agbledɔ01.db
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01/runs/classify/agbledɔ01-yolo_experiment-01/agbledɔ01
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50      3.17G     0.8275          9        640: 100% ━━━━━━━━━━━━ 615/615 1.9it/s 5:20
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 66/66 1.9it/s 34.2s
                   all      0.894          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50      3.87G     0.4464          9        640: 100% ━━━━━━━━━━━━ 615/615 1.9it/s 5:26
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 66/66 2.1it/s 31.8s
                   all      0.869          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50      3.

In [ ]:
#RUN_ID_FILE = PROJECT_ROOT / 'mlruns' / "run_id.txt"
#if RUN_ID_FILE.exists():
#    run_id = RUN_ID_FILE.read_text().strip()
#    print(f"🔄 Resume MLflow run_id={run_id}")
#    run = mlflow.start_run(run_id=run_id)
#    log_params = False
#else:
#    print("🆕 New MLflow run")
#    run = mlflow.start_run(run_name=run_name)
#    RUN_ID_FILE.write_text(run.info.run_id)
#    log_params = True

#with run:
#    if log_params:
#        mlflow.log_params(hyp_config)

"""with mlflow.start_run(run_name=run_name):
    if not resume_flag:
        mlflow.log_params(hyp_config)

    # Entraînement
    results = model.train(
        data=str(DATA_DIR),
        project="runs/classify",
        name=run_name,
        resume=resume_flag,
        epochs=hyp_config["epochs"],
        patience=hyp_config["patience"],
        batch=hyp_config["batch"],
        imgsz=hyp_config["imgsz"],
        device=hyp_config["device"],
        optimizer=hyp_config["optimizer"],
        lr0=hyp_config["lr0"],
        lrf=hyp_config["lrf"],
        weight_decay=hyp_config["weight_decay"],
        hsv_h=hyp_config["hsv_h"],
        hsv_s=hyp_config["hsv_s"],
        hsv_v=hyp_config["hsv_v"],
        degrees=hyp_config["degrees"],
        translate=hyp_config["translate"],
        scale=hyp_config["scale"],
        fliplr=hyp_config["fliplr"],
        mosaic=hyp_config["mosaic"],
        deterministic=True,
        seed=42,
        save=True
    )"""

#mlflow.end_run()